<a href="https://colab.research.google.com/github/CPTR295/Learning-Langchain/blob/main/Q%26A_Uisng_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q langchain_community langchain_chroma langchain_openai

In [9]:
!pip install -q wikipedia docx2txt pypdf chromadb

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
from langchain_community.document_loaders import WikipediaLoader,Docx2txtLoader,PyPDFLoader,TextLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

In [5]:
from google.colab import userdata
api_key = userdata.get("OPENROUTER_API_KEY")

In [6]:
import requests
from langchain_core.embeddings import Embeddings
from google.colab import userdata

api_key = userdata.get("OPENROUTER_API_KEY")

class OpenRouterNvidiaEmbeddings(Embeddings):
    def __init__(self, api_key):
        self.api_key = api_key
        self.url = "https://openrouter.ai/api/v1/embeddings"
        self.model = "nvidia/llama-nemotron-embed-vl-1b-v2:free"

    def _embed(self, texts, input_type):
        response = requests.post(
            self.url,
            headers={
                "Authorization": f"Bearer {self.api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": self.model,
                "input": [
                    {
                        "content": [
                            {"type": "text", "text": text}
                        ]
                    }
                    for text in texts
                ],
                "input_type": input_type,
                "encoding_format": "float",
            },
            timeout=120,
        )

        response.raise_for_status()
        data = response.json()["data"]
        data.sort(key=lambda item: item["index"])

        return [item["embedding"] for item in data]

    def embed_documents(self, texts):
        return self._embed(texts, "passage")

    def embed_query(self, text):
        return self._embed([text], "query")[0]


embeddings_model = OpenRouterNvidiaEmbeddings(api_key)

In [24]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)
vector_db = Chroma("tourist_info",embeddings_model)

In [25]:
wikipedia_loader = WikipediaLoader(query='Paestum')
wikipedia_chunks = text_splitter.split_documents(wikipedia_loader.load())
len(wikipedia_chunks)

118

In [26]:
vector_db.add_documents(wikipedia_chunks)

['027b0f83-7d76-4af7-a645-357180cf6d92',
 '23564695-d547-424a-b237-ccbc828c93ef',
 '66a630cd-c822-4919-98b6-282028c311e7',
 'cef45b16-bbca-4a6a-b24c-975385b347c4',
 'b5ad56a8-e530-4fd1-9bee-406f369e303c',
 '2262f677-a66e-44e2-a023-63c4d59ca363',
 '946806bc-d5ef-494b-a4fe-1c82634c60e7',
 'a176bc1d-cfff-4512-b4ff-ebfafc57ddd2',
 'a849df5e-4172-4fb1-9866-bfe6ad59336d',
 'e4565121-2dab-4b48-9d6c-70ea75e5c795',
 'cce13938-1056-47ec-9fe5-4f95ead752cf',
 '50da9028-9e4c-4c8a-8366-85ccc02c3743',
 '166b4efe-a59a-4943-9e59-7d289afc48cb',
 '43b4d82b-73e6-4ef1-b127-c500a7fb0274',
 '1c9a1ff0-6900-4276-a2a9-3791ee423c6b',
 'fd5d46a1-0363-48fa-aa29-af55b327696b',
 '6e390d37-059c-44b0-97ea-41ca102ae9bb',
 'bf99f5a2-e518-4d08-ac0b-9a409dc6bd85',
 '11fcc164-b10f-4bf0-bc21-28b6e721faa7',
 '59157956-3692-48d3-ac01-1d2f106ea001',
 'cea52cec-e862-4d4c-99f5-2592f3888892',
 'bb55f4a5-e3bb-42c8-bba6-cc471500e67c',
 '588250c9-fdb9-429e-b9ca-342b59146dac',
 '138052f6-7dd7-446f-bdf4-b3fd0fd2928c',
 '396a536c-078b-

In [27]:
def download_file(url,local_filename):
  response = requests.get(url)
  response.raise_for_status()

  with open(local_filename, 'wb') as f: #Dont use w because it wiriting text - not good for pdf and other files
    f.write(response.content)

  print(f"{local_filename}File downloaded successfully!")

In [28]:
url1 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/Paestum-Encyclopedia.txt'
url2 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/PaestumRevisited-StocholmsUniversitet.pdf'
url3 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/PaestumRevisited.pdf'
url4 = 'https://raw.githubusercontent.com/CPTR295/Learning-Langchain/refs/heads/main/Paestum-Britannica.docx'

download_file(url1,'Paestum-Encyclopedia.txt')
download_file(url1,'PaestumRevisited-StocholmsUniversitet.pdf')
download_file(url1,'PaestumRevisited.pdf')
download_file(url1,'Paestum-Britannica.docx')

Paestum-Encyclopedia.txtFile downloaded successfully!
PaestumRevisited-StocholmsUniversitet.pdfFile downloaded successfully!
PaestumRevisited.pdfFile downloaded successfully!
Paestum-Britannica.docxFile downloaded successfully!


In [31]:
#word_loader = Docx2txtLoader('/Paestum-Britannica.docx')
#word_chunks = text_splitter.split_documents(word_loader.load())
#len(word_chunks) #Not working not sure why

In [39]:
#pdf_loader = PyPDFLoader('./PaestumRevisited-StocholmsUniversitet.pdf')
#pdf_chunks = text_splitter.split_documents(pdf_loader.load())
#len(pdf_chunks)

In [41]:
pdf_loader = TextLoader(
    "PaestumRevisited-StocholmsUniversitet.pdf",
    encoding="utf-8"
)
documents = pdf_loader.load()
pdf_chunks = text_splitter.split_documents(documents)
print(len(pdf_chunks))

1


In [43]:
txt_loader = TextLoader("Paestum-Encyclopedia.txt")
txt_chunks = text_splitter.split_documents(
    txt_loader.load())
vector_db.add_documents(txt_chunks)

['c6bad5f4-31ed-4a9d-bef6-50c0382ad04d']

Remove Duplication

In [45]:
def split_and_import(loader):
  chunks = text_splitter.split_documents(loader.load())
  vector_db.add_documents(chunks)
  print(f'Ingested chunks created by the {loader}')

"""wikipedia_loader = WikipediaLoader(query="Paestum")
split_and_import(wikipedia_loader)

word_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")
split_and_import(word_loader)

pdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")
split_and_import(pdf_loader)

txt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")
split_and_import(txt_loader)"""

'wikipedia_loader = WikipediaLoader(query="Paestum")\nsplit_and_import(wikipedia_loader)\n\nword_loader = Docx2txtLoader("Paestum/Paestum-Britannica.docx")\nsplit_and_import(word_loader)\n\npdf_loader = PyPDFLoader("Paestum/PaestumRevisited.pdf")\nsplit_and_import(pdf_loader)\n\ntxt_loader = TextLoader("Paestum/Paestum-Encyclopedia.txt")\nsplit_and_import(txt_loader)'

Querying the vector store

In [46]:
query = "Where was Poseidonia and who renamed it to Paestum?"
results = vector_db.similarity_search(query, 4) # four clostest results
print(results)

[Document(id='23564695-d547-424a-b237-ccbc828c93ef', metadata={'summary': 'Paestum ( PEST-əm, US also  PEE-stəm, Latin: [ˈpae̯stũː]) was a major ancient Greek city on the coast of the Tyrrhenian Sea, in Magna Graecia. The ruins of Paestum are famous for their three ancient Greek temples in the Doric order dating from about 550 to 450 BCE that are in an excellent state of preservation. The city walls and amphitheatre are largely intact, and the bottom of the walls of many other structures remain, as well as paved roads. The site is open to the public, and there is a modern national museum within it, which also contains the finds from the associated Greek site of Foce del Sele.\nPaestum was established around 600 BCE by settlers from Sybaris, a Greek colony in southern Italy, under the name of Poseidonia (Ancient Greek: Ποσειδωνία). The city thrived as a Greek settlement for about two centuries, witnessing the development of democracy. In 400 BCE, the Lucanians seized the city. Romans to

In [47]:
len(results)

4

In [48]:
from langchain_core.prompts import PromptTemplate

reg_prompt_template = """Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use three sentences maximum and keep the answer as concise as possible.
{context}

Question: {question}
Helpful Answer:"""

rag_prompt = PromptTemplate(
    template=reg_prompt_template, input_variables=["context", "question"]
)

In [49]:
retriever = vector_db.as_retriever()

In [50]:
from langchain_core.runnables import RunnablePassthrough
question_feeder = RunnablePassthrough()

In [52]:
from langchain_openai import ChatOpenAI
chatbot = ChatOpenAI( #Many providers provide same api interface as openai including openrouter
    model="nvidia/nemotron-3-ultra-550b-a55b:free",
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1", )

In [53]:
rag_chain = {"context": retriever,
             "question": question_feeder} | rag_prompt | chatbot

In [54]:
def execute_chain(chain,question):
  answer = chain.invoke(question)
  return answer


In [55]:
question = """Where was Poseidonia and who renamed
it to Paestum. Also tell me the source."""
answer = execute_chain(rag_chain, question)
print(answer.content)

Paestum was an ancient Greek city on the coast of Magna Graecia in southern Italy (modern-day Campania). The Romans renamed it from Poseidonia to Paestum in 273 BCE after taking control from the Lucanians. Source: Wikipedia articles on Paestum.


In [57]:
print(answer)

content='Paestum was an ancient Greek city on the coast of Magna Graecia in southern Italy (modern-day Campania). The Romans renamed it from Poseidonia to Paestum in 273 BCE after taking control from the Lucanians. Source: Wikipedia articles on Paestum.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 379, 'prompt_tokens': 2163, 'total_tokens': 2542, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 317, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-ultra-550b-a55b:free', 'system_fingerprint': None, 'id': 'gen-1794474909-GY9Fr98sFw8pIZ1QDkpP', 'finish_reaso

In [60]:
question = """And then, what they do?
Tell me only if you know.
Also tell me the source"""
answer = execute_chain(rag_chain, question)
print(answer.content)


I don't know - your question "And then, what they do?" is too ambiguous. The documents mention sirens, Lucanians, and Poseidon, but it's unclear who "they" refers to or what specific action you're asking about.


Chatbot memory of message history

In [61]:
from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant, world-class expert in Roman and Greek history, especially in towns located in southern Italy. Provide interesting insights on local history and recommend places to visit with knowledgeable and engaging answers. Answer all questions to the best of your ability, but only use what has been provided in the context. If you don't know, just say you don't know. Use three sentences maximum and keep the answer as concise as possible."),
        ("placeholder", "{chat_history_messages}"),
        ("assistant", "{retrieved_context}"),
        ("human", "{question}"),
    ]
)

In [62]:
from langchain_community.chat_message_histories import ChatMessageHistory
chat_history_memeory = ChatMessageHistory()

/tmp/ipykernel_3042/2041829079.py:2: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  chat_history_memeory = ChatMessageHistory()


In [63]:
retriever = vector_db.as_retriever()
question_feeder = RunnablePassthrough()

In [64]:
def get_messages(x):
  return chat_history_memeory.messages

In [66]:
from langchain_core.runnables import RunnableLambda
rag_chain = {
    "retrieved_context": retriever,
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

In [67]:
def execute_chain_with_memory(chain,question):
  chat_history_memeory.add_user_message(question)
  answer = chain.invoke(question)
  chat_history_memeory.add_ai_message(answer)
  print(f"Full chat message history: {chat_history_memeory.messages}\n\n")
  return answer

In [68]:
question = """Where was Poseidonia and who renamed
it to Paestum? Also tell me the source."""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed \nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Paestum was located on the coast of the Tyrrhenian Sea in Magna Graecia (modern-day southern Italy, near Salerno). The Romans renamed it from Poseidonia to Paestum in 273 BCE after conquering it, though the Lucanians had earlier renamed it Paistom when they seized it in 400 BCE. Source: https://en.wikipedia.org/wiki/Paestum', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 295, 'prompt_tokens': 2232, 'total_tokens': 2527, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 206, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_in

In [70]:
question = """And then what did they do?
Also tell me the source"""
answer = execute_chain_with_memory(rag_chain, question)
print(answer.content)

Full chat message history: [HumanMessage(content='Where was Poseidonia and who renamed \nit to Paestum? Also tell me the source.', additional_kwargs={}, response_metadata={}), AIMessage(content='Paestum was located on the coast of the Tyrrhenian Sea in Magna Graecia (modern-day southern Italy, near Salerno). The Romans renamed it from Poseidonia to Paestum in 273 BCE after conquering it, though the Lucanians had earlier renamed it Paistom when they seized it in 400 BCE. Source: https://en.wikipedia.org/wiki/Paestum', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 295, 'prompt_tokens': 2232, 'total_tokens': 2527, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 206, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_in